
## `fact_reviews`

**Grain:** One record per customer review enriched with AI-generated sentiment and issue analysis.

| Column | Type | Description |
| --- | --- | --- |
| `review_timestamp` | `TIMESTAMP` | Timestamp when the review was submitted. |
| `review_id` | `STRING` | Unique identifier for the review. |
| `order_id` | `STRING` | Identifier of the associated order. |
| `customer_id` | `STRING` | Identifier of the customer who submitted the review. |
| `restaurant_id` | `STRING` | Identifier of the restaurant being reviewed. |
| `rating` | `INT` | Customer rating (typically 1–5). |
| `review_text` | `STRING` | Original review text provided by the customer. |
| `analysis_json` | `STRUCT` | AI-generated structured analysis of the review. |
| `sentiment` | `STRING` | Overall sentiment (`positive`, `neutral`, or `negative`). |
| `issue_delivery` | `BOOLEAN` | Indicates whether a delivery-related issue was identified. |
| `issue_delivery_reason` | `STRING` | Brief explanation of the delivery issue, if any. |
| `issue_food_quality` | `BOOLEAN` | Indicates whether a food quality issue was identified. |
| `issue_food_quality_reason` | `STRING` | Brief explanation of the food quality issue, if any. |
| `issue_pricing` | `BOOLEAN` | Indicates whether a pricing-related issue was identified. |
| `issue_pricing_reason` | `STRING` | Brief explanation of the pricing issue, if any. |
| `issue_portion_size` | `BOOLEAN` | Indicates whether a portion size issue was identified. |
| `issue_portion_size_reason` | `STRING` | Brief explanation of the portion size issue, if any. |

In [0]:
%sql
with model_response AS(
select *,
from_json(
    ai_query(
        'databricks-gpt-oss-20b',
        CONCAT(
            'Analyze the following review text and return ONLY a valid JSON object with this exact structure: ',
            '{"sentiment": "<positive/neutral/negative>", ',
            '"issue_delivery": <true/false>, ',
            '"issue_delivery_reason": "<reason or empty string>", ',
            '"issue_food_quality": <true/false>, ',
            '"issue_food_quality_reason": "<reason or empty string>", ',
            '"issue_pricing": <true/false>, ',
            '"issue_pricing_reason": "<reason or empty string>", ',
            '"issue_portion_size": <true/false>, ',
            '"issue_portion_size_reason": "<reason or empty string>"}. ',
            'Rules: sentiment must be exactly one of: positive, neutral, negative. ',
            'Each issue field is true/false only. ',
            'Each reason field should contain a brief explanation if the issue is true, otherwise empty string. ',
            'Review text: ', review_text
          )
    ),
    'STRUCT<sentiment:STRING, issue_delivery:BOOLEAN, issue_delivery_reason:STRING, issue_food_quality:BOOLEAN, issue_food_quality_reason:STRING, issue_pricing:BOOLEAN, issue_pricing_reason:STRING, issue_portion_size:BOOLEAN, issue_portion_size_reason:STRING>'
) AS analysis_json
from ws_restaurant_dbxproject.`01_bronze`.reviews
)

select
    review_timestamp,
    review_id,
    order_id,
    customer_id,
    restaurant_id,
    rating,
    review_text,
    analysis_json,
    analysis_json.sentiment as sentiment,
    analysis_json.issue_delivery as issue_delivery,
    analysis_json.issue_delivery_reason as issue_delivery_reason,
    analysis_json.issue_food_quality as issue_food_quality,
    analysis_json.issue_food_quality_reason as issue_food_quality_reason,
    analysis_json.issue_pricing as issue_pricing,
    analysis_json.issue_pricing_reason as issue_pricing_reason,
    analysis_json.issue_portion_size as issue_portion_size,
    analysis_json.issue_portion_size_reason as issue_portion_size_reason
from model_response;